# Intake: Gesmundo

The Gesmundo *et al.* miniaturised Suzuki informer screen -- 5 Buchwald
precatalysts over 12 cores x 10 monomers x 12 condition sets, 1,320 wells with
a measured conversion.

Source:
[cernaklab/medchem-reaction-miniaturization](https://github.com/cernaklab/medchem-reaction-miniaturization),
sheet `suzuki_informer` of `suzuki-ez.xlsx`.

**The design is unbalanced**, and it matters for reading the results: Aphos G3
carries four of the twelve (catalyst, base, cosolvent) condition sets and the
other four catalysts two each, so holding out Aphos G3 removes 441 of 1,320 rows
against ~215-226 for the others. Compare pooled metrics across methods, not
fold means.

With 5 catalysts LOLO is already a 5-fold partition, so `kfold_stratified_5`
**is** its matched control -- same splitter, same fold count, differing only in
whether the held-out catalyst was seen in training.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/gesmundo/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))

from gpc import prep
from gpc.data import load_bundle

pd.set_option("display.width", 220, "display.max_columns", 80)

OUT = HERE / "inputs"                  # the bundle this notebook writes

# ---- Editable -----------------------------------------------------------
DATASET      = "gesmundo"
DISPLAY_NAME = "Gesmundo"
TARGET       = "conversion"
GROUP        = "catalyst"
CATEGORICAL  = ["base", "cosolvent", "core", "monomer"]
SOURCE_URL   = "https://github.com/cernaklab/medchem-reaction-miniaturization"

# "adopt" reuses the verified table in this folder's bundle.
# "raw"   rebuilds from suzuki-ez.xlsx -- see the cell below.
SOURCE    = "adopt"
RAW_XLSX  = HERE / "raw" / "suzuki-ez.xlsx"
RAW_SHEET = "suzuki_informer"
# --------------------------------------------------------------------------
print(f"source mode: {SOURCE}")

## 1. Build the reactions table

The recorded decisions for this screen:

* **120 wells with no conversion value dropped**, leaving 1,320 of 1,440;
* `Chemistry` dropped (constant across all 1,440 wells), `Column` dropped
  (assay plumbing), `Condition Order` dropped (a run-order index, not a
  reagent);
* replicate rows sharing a feature row are **kept** -- 94 of them. Random folds
  can split a replicate pair across train and test; LOLO cannot. That is a
  reason to trust the LOLO numbers more than the k-fold ones here.

**On `SOURCE = "raw"`:** reading the sheet needs `openpyxl`, which is not in
this environment, and the original reshaping code is not in this repository.
The cell raises rather than guessing; `"adopt"` gives the identical verified
table.

In [ ]:
if SOURCE == "adopt":
    source_table = OUT / "reactions.csv"
    if not source_table.exists():
        raise FileNotFoundError(
            f"{source_table} not found. This is the seed table for the adopt "
            f"path; use SOURCE='raw' once the rebuild below is implemented.")
    reactions = pd.read_csv(source_table, keep_default_na=False)
    reactions[TARGET] = pd.to_numeric(reactions[TARGET])

elif SOURCE == "raw":
    # pd.read_excel(RAW_XLSX, sheet_name=RAW_SHEET) needs openpyxl:
    #     pip install openpyxl
    # then drop the wells with no conversion and the three non-reagent columns
    # listed above, rename to the canonical schema, and continue below.
    raise NotImplementedError(
        "The Gesmundo raw rebuild is not in this repository, and reading the "
        "sheet needs openpyxl. The decisions it must reproduce are listed "
        "above; the resulting frame goes through prep.add_row_ids and "
        "prep.attach_kraken exactly as the adopt path does.")
else:
    raise ValueError(f"SOURCE must be 'adopt' or 'raw', not {SOURCE!r}")

print(f"{len(reactions)} wells x {reactions.shape[1]} columns")
display(reactions.head(3))

## 2. The catalyst mapping

These are precatalysts (G2/G3 palladacycles), so the mapping is to the Kraken
entry for the **phosphine they carry** -- that is the chemistry the descriptors
describe. `Aphos G3` maps to AmPhos (216), `tBu3P G2` to P(tBu)3 (8).

In [ ]:
# ---- Editable -----------------------------------------------------------
MAPPING = {"Aphos G3": 216, "RuPhos G3": 4, "Xphos G3": 1,
           "tBu3P G2": 8, "tBuXphos G3": 90}
# --------------------------------------------------------------------------

reactions = prep.attach_kraken(reactions.drop(columns=["kraken_id"], errors="ignore"),
                               GROUP, MAPPING)
if "row_id" not in reactions:
    reactions = prep.add_row_ids(reactions)

identifiers = prep.kraken_identifiers().set_index("id")
display(pd.DataFrame([{"catalyst": name, "kraken_id": kid,
                       "kraken_name": identifiers["ligand"].get(kid, "?")}
                      for name, kid in MAPPING.items()]))

## 4. Describe it, then validate

The table below is the record of what this bundle claims about itself: rows and
target statistics per group. Read it before writing -- an unbalanced design
shows up here, and it changes how the LOLO folds should be read (their sizes
follow these counts).

Aphos G3's row count is roughly double the others -- that is the unbalanced
design, not a data error.

In [ ]:
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

In [ ]:
# ---- Editable: what this dataset IS -------------------------------------
cfg = prep.bundle_config(
    dataset=DATASET,
    display_name=DISPLAY_NAME,
    target=TARGET,
    group=GROUP,
    categorical=CATEGORICAL,
    reactions=reactions,
    source=SOURCE_URL,
    # models/methods default sensibly: all five sections, and the method list
    # whose matched control has this screen's own group count. Pass explicit
    # lists here to override.
)
# --------------------------------------------------------------------------

display(pd.json_normalize(cfg["data"]).T.rename(columns={0: "value"}))
print("methods:", cfg["evaluation"]["methods"])

# Every check load_bundle will make, run here so a bad bundle fails at the
# point it was built rather than on the cluster three hours into a job.
display(prep.check_bundle(reactions, cfg, prep.kraken_features()))

## 5. Write the bundle

`write_bundle` writes `reactions.csv`, the Kraken descriptor rows for the
ligands this screen uses, `ligand_mapping.csv`, the copied reference PCA and
`config.json`. The PCA is **copied, never refitted** -- PC1..PC4 have to mean
the same axes in every dataset or `pc_top` and `pc_scores` stop being
comparable, which is the point of running them.

In [ ]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING)

# Read it straight back through the engine's own loader: if this succeeds, the
# bundle is usable by `gpc train` exactly as written.
back, ligands, reference, prepared = load_bundle(bundle)
print(f"\nreloaded: {len(back)} rows, {back[cfg['data']['group']].nunique()} "
      f"{cfg['data']['group']}s, {len(ligands)} descriptor rows, "
      f"PCA over {len(reference.columns)} descriptors")
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))
print("\nNext: build_run.ipynb in the hub root, and pick this dataset.")